In [ ]:
from pyspark.sql import SparkSession, Window

In [ ]:
spark =SparkSession.builder \
         .appName("gold_layer") \
         .master('local[*]') \
         .config('spark.sql.shuffle.partitions', '8') \
         .config('spark.driver.memory', '2g') \
         .config('spark.sql.parquet.writeLegacyFormat', 'true') \
         .enableHiveSupport() \
         .getOrCreate()

In [ ]:
from datetime import date
import pyspark.sql.functions as F

spark.sql("SET hive.exec.dynamic.partition=true")
spark.sql("SET hive.exec.dynamic.partition.mode=nonstrict")

db = "looker_ecommerce_dwh"
silver = "hdfs:///Data/Silver/Cleaned_Looker_Ecommerce_BigQuery_Dataset"

def date_key(c):
    return F.date_format(c, "yyyyMMdd").cast("int")

%run ./utils.ipynb

dim_users

In [ ]:
users = (spark.read.parquet(f"{silver}/users")
         .withColumn("traffic_source", F.coalesce("traffic_source", F.lit("Unknown")))
         .withColumn("registered_date_key", date_key(F.col("registered_at")))
         # the first version of each user starts on the day they registered (not on the load date)
         .withColumn("first_start", F.to_date("registered_at"))
         .drop("registered_at"))
scd2_load(users,
          f"{db}.dim_users",
          business_key="user_id",
          tracked_cols=["first_name","last_name","age","state", "street_address", "city", "country"],
          spark=spark,surrogate_key="user_key", first_start_col="first_start")

# Unknown member (user_key 0): target for anonymous events and unmatched users.
if spark.table(f"{db}.dim_users").filter("user_key = 0").limit(1).count() == 0:
    unknown_user = spark.createDataFrame(
        [(0, -1, "Unknown", "Unknown", "Unknown", None, "Unknown", "Unknown",
          "Unknown", "Unknown", "Unknown", None, date.today(), None, True, "Unknown")],
        "user_key BIGINT, user_id INT, first_name STRING, last_name STRING, email STRING, "
        "age INT, gender STRING, state STRING, street_address STRING, city STRING, "
        "country STRING, registered_date_key INT, start_date DATE, end_date DATE, "
        "is_current BOOLEAN, traffic_source STRING")
    unknown_user.write.mode("append").insertInto(f"{db}.dim_users")

dim_products

In [ ]:
products = spark.read.parquet(f"{silver}/products")

distribution_centers = spark.read.parquet(f"{silver}/distribution_centers")

# the first version of each product starts on the day of its first order (not on the load date)
first_order = (spark.read.parquet(f"{silver}/order_items")
               .groupBy("product_id")
               .agg(F.to_date(F.min("created_at")).alias("first_start")))

dim_products_df = (products.join(distribution_centers, on="distribution_center_id", how="left")
    .join(first_order, on="product_id", how="left")
    .select("product_id","product_name","brand", "category", "department", "cost", "retail_price",
            "distribution_center_name", "distribution_center_state", "first_start"))


scd2_load(dim_products_df,
          f"{db}.dim_products",
          business_key="product_id",
          tracked_cols=["product_name","cost", "retail_price","distribution_center_name", "distribution_center_state"],
          spark=spark,surrogate_key="product_key", first_start_col="first_start")

if spark.table(f"{db}.dim_products").filter("product_key = 0").limit(1).count() == 0:
    unknown_product = spark.createDataFrame(
        [(0, -1, "Unknown", "Unknown", "Unknown", "Unknown", None, None,
          "Unknown", "Unknown", date.today(), None, True)],
        "product_key BIGINT, product_id INT, product_name STRING, brand STRING, category STRING, "
        "department STRING, cost DECIMAL(10,2), retail_price DECIMAL(10,2), "
        "distribution_center_name STRING, distribution_center_state STRING, "
        "start_date DATE, end_date DATE, is_current BOOLEAN")
    unknown_product.write.mode("append").insertInto(f"{db}.dim_products")

dim_order_status

In [ ]:
order_status_df = spark.createDataFrame(
    [(0, "Processing"), (1, "Shipped"), (2, "Complete"), (3, "Returned"), (4, "Cancelled"), (-1, "Unknown")],
    "status_id INT, status_name STRING")

simple_load(order_status_df, f"{db}.dim_order_status", business_key="status_id", spark=spark)

dim_date

In [ ]:
order_items = spark.read.parquet(f"{silver}/order_items")
events      = spark.read.parquet(f"{silver}/events")
users       = spark.read.parquet(f"{silver}/users")

b1 = order_items.agg(
        F.min("created_at").alias("low"),
        F.max(F.greatest("created_at", "shipped_at", "delivered_at", "returned_at")).alias("high")
     ).collect()[0]

b2 = events.agg(F.min("created_at").alias("low"),
                F.max("created_at").alias("high")).collect()[0]

b3 = users.agg(F.min("registered_at").alias("low"),
               F.max("registered_at").alias("high")).collect()[0]

lo = min(b1["low"], b2["low"], b3["low"])
hi = max(b1["high"], b2["high"], b3["high"])

last_year = max(hi.year, date.today().year) + 1
start, end = date(lo.year, 1, 1), date(last_year, 12, 31)
n_days = (end - start).days + 1
print(start, end, n_days)

dim_date_df = (spark.range(n_days)
    .select(F.expr(f"date_add(to_date('{start.isoformat()}'), cast(id as int))").alias("full_date"))
    .select(
        F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
        "full_date",
        F.dayofmonth("full_date").alias("day_of_month"),
        F.dayofweek("full_date").alias("day_of_week"),
        F.date_format("full_date", "EEEE").alias("day_name"),
        F.dayofyear("full_date").alias("day_of_year"),
        F.weekofyear("full_date").alias("week_of_year"),
        F.month("full_date").alias("month_number"),
        F.date_format("full_date", "MMMM").alias("month_name"),
        F.quarter("full_date").alias("calendar_quarter"),
        F.year("full_date").alias("calendar_year"),
        F.dayofweek("full_date").isin(1, 7).alias("is_weekend"),
    ))

simple_load(dim_date_df, f"{db}.dim_date", business_key="date_key", spark=spark)

fact_order_items

In [ ]:
order_items = spark.read.parquet(f"{silver}/order_items")

order_items_keyed = attach_product_key(
    attach_user_key(order_items, "created_at", "order_item_id", spark, db),
    "created_at", "order_item_id", spark, db)

fact_order_items_df = (order_items_keyed
    .select(
        "order_item_id", "order_id",
        "user_key",
        "product_key",
        "status_id",
        "sale_price",
        date_key("created_at").alias("created_date_key"),
        date_key("shipped_at").alias("shipped_date_key"),
        date_key("delivered_at").alias("delivered_date_key"),
        date_key("returned_at").alias("returned_date_key"),
        F.year("created_at").alias("order_year")))

upsert_fact(df=fact_order_items_df,
            table_name=f"{db}.fact_order_items",
            business_key="order_item_id",
            partition_cols=["order_year"],
            spark=spark
)

fact_events

In [ ]:
events = spark.read.parquet(f"{silver}/events")

fact_events_df = (attach_user_key(events, "created_at", "event_id", spark, db)
    .select(
        "event_id",
        "user_key",
        date_key("created_at").alias("created_date_key"),
        "session_id", "sequence_number", "event_type", "uri", "browser", "traffic_source",
        "city", "state",
        F.year("created_at").alias("event_year"),
        F.month("created_at").alias("event_month")))

simple_load(
    df=fact_events_df,
    table_name=f"{db}.fact_events",
    business_key="event_id",
    partition_cols=["event_year", "event_month"],
    spark=spark
)